In [1]:
//dependencias
import $ivy.`org.apache.spark::spark-sql:4.1.1`
import $ivy.`io.delta::delta-spark_4.1:4.2.0`

import $ivy.$
import $ivy.$

In [2]:
//creamos Spark Session
import org.apache.spark.sql.SparkSession

val spark = SparkSession.builder()
  .appName("ProyectoBigData")
  .master("local[*]")
  .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
  .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
  .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/05/11 18:57:43 WARN Utils: Your hostname, SamuraiLoco, resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/05/11 18:57:43 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
26/05/11 18:57:43 INFO SparkContext: Running Spark version 4.1.1
26/05/11 18:57:43 INFO SparkContext: OS info Linux, 6.6.114.1-microsoft-standard-WSL2, amd64
26/05/11 18:57:43 INFO SparkContext: Java version 17.0.18+8-Ubuntu-124.04.1
26/05/11 18:57:43 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/05/11 18:57:43 INFO ResourceUtils: ==============================================================
26/05/11 18:57:43 INFO ResourceUtils: No custom resources configured for spark.driver.
26/05/11 18:57:43 INFO ResourceUtils: ==============================================================
26/05/11 18:57:43

import org.apache.spark.sql.SparkSession
spark: SparkSession = org.apache.spark.sql.classic.SparkSession@2466d35a

In [3]:
//Verificamos spark
spark.version

res3: String = "4.1.1"

In [4]:
//paths
val basePath = "/mnt/c/Users/danie/OneDrive/Desktop/Proyectos/Proyecto_final_BIGDATA"

val ordersPath = basePath + "/datos/orders_raw.csv"

val marketingPath = basePath + "/datos/marketing_campaigns.json"

val bronzeOrdersPath = basePath + "/delta/bronze/orders"

val bronzeMarketingPath = basePath + "/delta/bronze/marketing_campaigns"

basePath: String = "/mnt/c/Users/danie/OneDrive/Desktop/Proyectos/Proyecto_final_BIGDATA"
ordersPath: String = "/mnt/c/Users/danie/OneDrive/Desktop/Proyectos/Proyecto_final_BIGDATA/datos/orders_raw.csv"
marketingPath: String = "/mnt/c/Users/danie/OneDrive/Desktop/Proyectos/Proyecto_final_BIGDATA/datos/marketing_campaigns.json"
bronzeOrdersPath: String = "/mnt/c/Users/danie/OneDrive/Desktop/Proyectos/Proyecto_final_BIGDATA/delta/bronze/orders"
bronzeMarketingPath: String = "/mnt/c/Users/danie/OneDrive/Desktop/Proyectos/Proyecto_final_BIGDATA/delta/bronze/marketing_campaigns"

In [5]:
//leemos csv raw
val ordersRawDF = spark.read
  .option("header", "true")
  .option("inferSchema", "true")
  .csv(ordersPath)

ordersRawDF.show(5)

+----------+----------+----+-----+-------+------+-----------+---------------+----------------+-------------+-------+-------------+------------+--------------------+----------+--------+--------+-------+-------+-------+---------------+-------------+---------------+-------------+--------------+------------+
|  Order_ID|Order_Date|Year|Month|Quarter|Season|Customer_ID|Customer_Gender|Customer_Segment|       Region|Country|     Category|Sub_Category|        Product_Name|Unit_Price|Quantity|Discount|Revenue|   Cost| Profit|Profit_Margin_%|Shipping_Cost|Shipping_Method|Shipping_Days|Payment_Method|Order_Status|
+----------+----------+----+-----+-------+------+-----------+---------------+----------------+-------------+-------+-------------+------------+--------------------+----------+--------+--------+-------+-------+-------+---------------+-------------+---------------+-------------+--------------+------------+
|ORD-000001|2023-01-18|2023|    1|     Q1|Winter| CUST-02791|           Male|     

ordersRawDF: org.apache.spark.sql.package.DataFrame = [Order_ID: string, Order_Date: date ... 24 more fields]

In [6]:
//ver schema
ordersRawDF.printSchema()

root
 |-- Order_ID: string (nullable = true)
 |-- Order_Date: date (nullable = true)
 |-- Year: integer (nullable = true)
 |-- Month: integer (nullable = true)
 |-- Quarter: string (nullable = true)
 |-- Season: string (nullable = true)
 |-- Customer_ID: string (nullable = true)
 |-- Customer_Gender: string (nullable = true)
 |-- Customer_Segment: string (nullable = true)
 |-- Region: string (nullable = true)
 |-- Country: string (nullable = true)
 |-- Category: string (nullable = true)
 |-- Sub_Category: string (nullable = true)
 |-- Product_Name: string (nullable = true)
 |-- Unit_Price: double (nullable = true)
 |-- Quantity: integer (nullable = true)
 |-- Discount: double (nullable = true)
 |-- Revenue: double (nullable = true)
 |-- Cost: double (nullable = true)
 |-- Profit: double (nullable = true)
 |-- Profit_Margin_%: double (nullable = true)
 |-- Shipping_Cost: double (nullable = true)
 |-- Shipping_Method: string (nullable = true)
 |-- Shipping_Days: integer (nullable = true)

In [7]:
//leer Json
val marketingDF = spark.read
  .option("multiline", "true")
  .json(marketingPath)

marketingDF.show()

+-------------+--------------+-----------------+--------------+
|campaign_cost|      category|marketing_channel|target_country|
+-------------+--------------+-----------------+--------------+
|         8500|   Electronics|        Instagram| United States|
|         6200|      Clothing|           TikTok|United Kingdom|
|         7300|Home & Kitchen|       Google Ads|       Germany|
|         4100|        Beauty|     Facebook Ads|        France|
|         5600|        Sports|          YouTube|         Spain|
|         8500|   Electronics|        Instagram| United States|
|         3000|          NULL|           TikTok|         Italy|
|   error_cost|          Toys|             NULL|        Canada|
+-------------+--------------+-----------------+--------------+



marketingDF: org.apache.spark.sql.package.DataFrame = [campaign_cost: string, category: string ... 2 more fields]

In [8]:
//importamos funciones spark
import org.apache.spark.sql.functions._

import org.apache.spark.sql.functions._

In [9]:
//crear bronze para orders csv
val ordersBronzeDF = ordersRawDF
  .withColumn("ingestion_timestamp", current_timestamp())
  .withColumn("source_file", lit("orders_raw.csv"))

ordersBronzeDF.show(5)

+----------+----------+----+-----+-------+------+-----------+---------------+----------------+-------------+-------+-------------+------------+--------------------+----------+--------+--------+-------+-------+-------+---------------+-------------+---------------+-------------+--------------+------------+--------------------+--------------+
|  Order_ID|Order_Date|Year|Month|Quarter|Season|Customer_ID|Customer_Gender|Customer_Segment|       Region|Country|     Category|Sub_Category|        Product_Name|Unit_Price|Quantity|Discount|Revenue|   Cost| Profit|Profit_Margin_%|Shipping_Cost|Shipping_Method|Shipping_Days|Payment_Method|Order_Status| ingestion_timestamp|   source_file|
+----------+----------+----+-----+-------+------+-----------+---------------+----------------+-------------+-------+-------------+------------+--------------------+----------+--------+--------+-------+-------+-------+---------------+-------------+---------------+-------------+--------------+------------+-----------

ordersBronzeDF: org.apache.spark.sql.package.DataFrame = [Order_ID: string, Order_Date: date ... 26 more fields]

In [10]:
//crear bronze para marketing json
val marketingBronzeDF = marketingDF
  .withColumn("ingestion_timestamp", current_timestamp())
  .withColumn("source_file", lit("marketing_campaigns.json"))

marketingBronzeDF.show()

+-------------+--------------+-----------------+--------------+--------------------+--------------------+
|campaign_cost|      category|marketing_channel|target_country| ingestion_timestamp|         source_file|
+-------------+--------------+-----------------+--------------+--------------------+--------------------+
|         8500|   Electronics|        Instagram| United States|2026-05-11 19:00:...|marketing_campaig...|
|         6200|      Clothing|           TikTok|United Kingdom|2026-05-11 19:00:...|marketing_campaig...|
|         7300|Home & Kitchen|       Google Ads|       Germany|2026-05-11 19:00:...|marketing_campaig...|
|         4100|        Beauty|     Facebook Ads|        France|2026-05-11 19:00:...|marketing_campaig...|
|         5600|        Sports|          YouTube|         Spain|2026-05-11 19:00:...|marketing_campaig...|
|         8500|   Electronics|        Instagram| United States|2026-05-11 19:00:...|marketing_campaig...|
|         3000|          NULL|           TikTo

marketingBronzeDF: org.apache.spark.sql.package.DataFrame = [campaign_cost: string, category: string ... 4 more fields]

In [ ]:
//guardar nronze en delta
ordersBronzeDF.write
  .format("delta")
  .mode("overwrite")
  .save(bronzeOrdersPath)

marketingBronzeDF.write
  .format("delta")
  .mode("overwrite")
  .save(bronzeMarketingPath)

In [12]:
//verificar
val ordersBronzeCheck = spark.read
  .format("delta")
  .load(bronzeOrdersPath)

val marketingBronzeCheck = spark.read
  .format("delta")
  .load(bronzeMarketingPath)

println(s"Orders Bronze: ${ordersBronzeCheck.count()} filas")
println(s"Marketing Bronze: ${marketingBronzeCheck.count()} filas")

Orders Bronze: 10000 filas
Marketing Bronze: 8 filas


ordersBronzeCheck: org.apache.spark.sql.package.DataFrame = [Order_ID: string, Order_Date: date ... 26 more fields]
marketingBronzeCheck: org.apache.spark.sql.package.DataFrame = [campaign_cost: string, category: string ... 4 more fields]